### f) Генерация выборки и базовые статистики

In [20]:
import numpy as np
import scipy.stats as st

np.random.seed(42)

n = 100
beta = 0.95
theta = 10

X = st.uniform(loc=theta, scale=theta).rvs(size=n)
X

array([13.74540119, 19.50714306, 17.31993942, 15.98658484, 11.5601864 ,
       11.5599452 , 10.58083612, 18.66176146, 16.01115012, 17.08072578,
       10.20584494, 19.69909852, 18.32442641, 12.12339111, 11.81824967,
       11.8340451 , 13.04242243, 15.24756432, 14.31945019, 12.9122914 ,
       16.11852895, 11.39493861, 12.92144649, 13.66361843, 14.56069984,
       17.85175961, 11.99673782, 15.14234438, 15.92414569, 10.46450413,
       16.07544852, 11.70524124, 10.65051593, 19.48885537, 19.65632033,
       18.08397348, 13.04613769, 10.97672114, 16.84233027, 14.40152494,
       11.22038235, 14.9517691 , 10.34388521, 19.09320402, 12.58779982,
       16.62522284, 13.11711076, 15.20068021, 15.46710279, 11.84854456,
       19.69584628, 17.75132823, 19.39498942, 18.9482735 , 15.97899979,
       19.21874235, 10.88492502, 11.95982862, 10.45227289, 13.25330331,
       13.8867729 , 12.71349032, 18.28737509, 13.56753327, 12.8093451 ,
       15.42696083, 11.40924225, 18.02196981, 10.74550644, 19.86

In [21]:
mx = np.mean(X)
xmax = np.max(X)
theta1 = 2/3 * mx
theta2 = (n+1) * xmax / (2*n+1)
mu2 = np.var(X)

### d, e) Точный и асимптотический доверительные интервалы

In [22]:
accurate_left = xmax / (((1+beta)/2)**(1/n) + 1)
accurate_right = xmax / (((1-beta)/2)**(1/n) + 1)
accurate_l = accurate_right - accurate_left

print("Точный доверительный интервал для theta:", accurate_left, "< theta <", accurate_right)
print("l =", accurate_l)

t1 = st.norm.ppf((1-beta)/2)
t2 = st.norm.ppf((1+beta)/2)
asymptotic_left = theta1 - 2/3 * np.sqrt(mu2/n) * t2
asymptotic_right = theta1 - 2/3 * np.sqrt(mu2/n) * t1
asymp_l = asymptotic_right - asymptotic_left

print("Асимптотический доверительный интервал для theta:", asymptotic_left, "< theta <", asymptotic_right)
print("l =", asymp_l)

Точный доверительный интервал для theta: 9.935692273544552 < theta < 10.117648567228192
l = 0.18195629368364052
Асимптотический доверительный интервал для theta: 9.414441046729793 < theta < 10.187968864979668
l = 0.7735278182498746


### g) Бутстраповские доверительные интервалы

In [23]:
N = 1000
idx1 = int((1 - beta)/2 * N - 1)
idx2 = int((1 + beta)/2 * N - 1)

def bootstrap_OMM(x, N):
    arr = []
    n_size = len(x)
    for _ in range(N):
        sample = np.random.choice(x, size=n_size, replace=True)
        arr.append(2/3 * np.mean(sample) - theta1)
    return sorted(np.array(arr))

bs_arr_omm = bootstrap_OMM(X, N)
bs_omm_left = theta1 - bs_arr_omm[idx2]
bs_omm_right = theta1 - bs_arr_omm[idx1]
bs_omm_l = bs_omm_right - bs_omm_left

print("Непараметрический бутстраповский доверительный интервал для theta(ОММ):", bs_omm_left, "< theta <", bs_omm_right)
print("l =", bs_omm_l)

def bootstrap_OMP(x, N):
    arr = []
    n_size = len(x)
    for _ in range(N):
        sample = np.random.choice(x, size=n_size, replace=True)
        arr.append((n_size+1) * np.max(sample) / (2*n_size+1) - theta2)
    return sorted(np.array(arr))

bs_arr_omp = bootstrap_OMP(X, N)
bs_omp_left = theta2 - bs_arr_omp[idx2]
bs_omp_right = theta2 - bs_arr_omp[idx1]
bs_omp_l = bs_omp_right - bs_omp_left

print("Непараметрический бутстраповский доверительный интервал для theta(ОМP):", bs_omp_left, "< theta <", bs_omp_right)
print("l =", bs_omp_l)

Непараметрический бутстраповский доверительный интервал для theta(ОММ): 9.407475257533175 < theta < 10.176736867927382
l = 0.7692616103942065
Непараметрический бутстраповский доверительный интервал для theta(ОМP): 9.983859731176729 < theta < 10.09066297774995
l = 0.1068032465732216


### h) Сравнение результатов

In [24]:
sort_lens = sorted([
    (accurate_l, "Точный"),
    (asymp_l, "Асимптотический"),
    (bs_omm_l, "Бутстрап ОММ"),
    (bs_omp_l, "Бутстрап ОМП")
])

print("Рейтинг доверительных интервалов по ширине:")
for i in range(len(sort_lens)):
    print(f"{i+1}) {sort_lens[i][1]} (l = {np.round(sort_lens[i][0], 3)})")

Рейтинг доверительных интервалов по ширине:
1) Бутстрап ОМП (l = 0.107)
2) Точный (l = 0.182)
3) Бутстрап ОММ (l = 0.769)
4) Асимптотический (l = 0.774)
